# 02 — Retrieval + Embedding Shootout (bge-m3 vs zembed-1)
Ponytail: one file, no DB. Run on server.

In [ ]:
import requests
MAILTO="ntminhduy1408@gmail.com"
q="CAN intrusion detection"
r=requests.get("https://api.openalex.org/works", params={"search":q,"per-page":5,"mailto":MAILTO}, timeout=60)
print(r.status_code)
[
 print('-',w.get('display_name'),w.get('publication_year'))
 for w in r.json().get('results',[])
]

In [ ]:
# ponytail: 4 pos + 3 neg only, enough to judge separation
POS=["Knowledge distillation GNN Transformer to tiny ECU student <10K params for CAN IDS",
 "Small language models MiniLM DistilBERT convert CAN traffic to tokens for intrusion detection",
 "Open-set few-shot cross-dataset CAN IDS Car-Hacking to CIC-IoV-2024",
 "Vision Transformer on CAN image recurrence plots for intrusion detection"]
NEG=["HairCLIP text image hair editing StyleGAN",
 "AlphaFold protein 3D structure prediction",
 "LLM summarization of legal contracts"]
THESIS="deep learning for in-vehicle CAN intrusion detection"
print(len(POS),len(NEG))

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
def scores(name):
    m=SentenceTransformer(name, trust_remote_code=True)
    # zembed wants prefixes; harmless for bge
    qt=m.encode(['query: '+THESIS], normalize_embeddings=True)
    P=m.encode(['passage: '+t for t in POS], normalize_embeddings=True)
    N=m.encode(['passage: '+t for t in NEG], normalize_embeddings=True)
    import numpy as np
    ps=(P@qt.T).ravel(); ns=(N@qt.T).ravel()
    print(f"{name}\npos={sorted(round(float(x),3) for x in ps)}\nneg={sorted(round(float(x),3) for x in ns)}")
    print(f"gap={min(ps)-max(ns):.3f} (pos_min - neg_max, want >0.05)")
for n in ["BAAI/bge-m3","zeroentropy/zembed-1-embedding"]:
    scores(n)

Judge: bigger gap wins. If zembed gap > bge gap → lock `zembed-1 @1536 int8`. Paste gaps back.